In [2]:
# Load env variables and create client
from anthropic import Anthropic
from dotenv import load_dotenv
from IPython.display import display
from IPython.display import Markdown
import debugpy
import json
import os
import pprint, shutil
import textwrap

load_dotenv()

if not os.getenv("ANTHROPIC_API_KEY"):
    exit("No API key found in environment variables")

client = Anthropic()
model = "claude-haiku-4-5"

In [3]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.with_options(timeout=30.0).messages.create(**params)
    return message.content[0].text

def myprint(messages):
    width = shutil.get_terminal_size().columns
    for line in json.dumps(messages, indent=2, ensure_ascii=False).splitlines():
        indent = len(line) - len(line.lstrip())
        print(textwrap.fill(line, width=width, subsequent_indent=' ' * (indent + 2)))

In [4]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [7]:
dataset = generate_dataset()

myprint(dataset)
json.dump(dataset, open("dataset.json", "w"), indent=2, ensure_ascii=False)

[
  {
    "task": "Write a regular expression to validate AWS S3 bucket names. S3
      bucket names must be between 3-63 characters, contain only lowercase
      letters, numbers, hyphens, and periods, start and end with a letter or
      number, and cannot contain consecutive hyphens or periods."
  },
  {
    "task": "Write a Python function that takes an AWS CloudFormation template
      (as a dictionary) and returns a list of all resource logical IDs that have
      a DependsOn property."
  },
  {
    "task": "Write a JSON object that represents a valid AWS IAM policy allowing
      read-only access to all objects in a specific S3 bucket named 'my-data-
      bucket'."
  }
]
